In [1]:
import os
import numpy as np
import pandas as pd
from scipy import signal
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Rutas
DATA_DIR = '/home/manu/TFG2/schizophrenia_dataset'
OUTPUT_DIR = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_Schizophrenia'

# Parámetros
FS = 1024  # Frecuencia de muestreo
FREQ_BAND = (8, 12)  # Banda Alpha
CONDITION = 1  # Condición a usar (button press + tone)

# Canales EEG (64 canales, excluyendo oculares)
EEG_CHANNELS = [
    'Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1',
    'C1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9',
    'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz', 'Fpz', 'Fp2', 'AF8', 'AF4',
    'AFz', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz',
    'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'P8', 'P10',
    'PO8', 'PO4', 'O2'
]

print(f"Canales EEG: {len(EEG_CHANNELS)}")

Canales EEG: 64


In [3]:
# Leer demografía
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'demographic.csv'))
demo_df.columns = demo_df.columns.str.strip()  # Limpiar espacios

# Separar por grupo
hc_subjects = demo_df[demo_df['group'] == 0]['subject'].tolist()
sz_subjects = demo_df[demo_df['group'] == 1]['subject'].tolist()

print(f"HC (Healthy Controls): {len(hc_subjects)} sujetos")
print(f"SZ (Schizophrenia): {len(sz_subjects)} sujetos")
print(f"Total: {len(hc_subjects) + len(sz_subjects)} sujetos")

HC (Healthy Controls): 32 sujetos
SZ (Schizophrenia): 49 sujetos
Total: 81 sujetos


In [4]:
# División 80/20 aproximada
# HC: 32 sujetos -> 26 train, 6 test
# SZ: 49 sujetos -> 39 train, 10 test

np.random.seed(42)

# Shuffle y dividir HC
hc_shuffled = np.random.permutation(hc_subjects).tolist()
hc_train = hc_shuffled[:26]
hc_test = hc_shuffled[26:]

# Shuffle y dividir SZ
sz_shuffled = np.random.permutation(sz_subjects).tolist()
sz_train = sz_shuffled[:39]
sz_test = sz_shuffled[39:]

print(f"Training: {len(hc_train)} HC + {len(sz_train)} SZ = {len(hc_train) + len(sz_train)} sujetos")
print(f"Test: {len(hc_test)} HC + {len(sz_test)} SZ = {len(hc_test) + len(sz_test)} sujetos")

# Guardar división
train_subjects = {'HC': hc_train, 'SZ': sz_train}
test_subjects = {'HC': hc_test, 'SZ': sz_test}

Training: 26 HC + 39 SZ = 65 sujetos
Test: 6 HC + 10 SZ = 16 sujetos


In [ ]:
def compute_coherence_matrix(data, fs, freq_band):
    """
    Calcula la matriz de coherencia cuadrada (squared coherence) para una época.
    """
    n_channels = data.shape[0]
    coh_matrix = np.zeros((n_channels, n_channels))
    
    # Calcular coherencia entre cada par de canales
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                coh_matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=fs, nperseg=min(256, len(data[i])))
                # Promediar coherencia en la banda de frecuencia
                freq_mask = (f >= freq_band[0]) & (f <= freq_band[1])
                if np.any(freq_mask):
                    coh_value = np.mean(Cxy[freq_mask])
                else:
                    coh_value = 0.0
                coh_matrix[i, j] = coh_value
                coh_matrix[j, i] = coh_value  # Simétrica
    
    return coh_matrix


def select_channels(n_target, n_total=64):
    """
    Selecciona canales equiespaciados.
    """
    if n_target >= n_total:
        return list(range(n_total))
    indices = np.linspace(0, n_total - 1, n_target, dtype=int)
    return indices.tolist()

In [ ]:
def process_subject(subject_id, condition=1):
    """
    Procesa todos los trials de un sujeto para una condición.
    """
    # Buscar archivo del sujeto
    subject_file = os.path.join(DATA_DIR, f'{subject_id}.csv', f'{subject_id}.csv')
    
    if not os.path.exists(subject_file):
        print(f"  Archivo no encontrado: {subject_file}")
        return []
    
    # Leer datos
    df = pd.read_csv(subject_file, header=None)
    
    # Columnas: subject, trial, condition, sample, + 64 EEG + 6 oculares
    # Los canales EEG están en columnas 4 a 67 (índices 4:68)
    
    # Filtrar por condición
    df_cond = df[df.iloc[:, 2] == condition]
    
    if len(df_cond) == 0:
        print(f"  No hay datos para condición {condition}")
        return []
    
    # Obtener trials únicos
    trials = df_cond.iloc[:, 1].unique()
    
    matrices = []
    
    for trial in trials:
        # Extraer datos del trial
        trial_data = df_cond[df_cond.iloc[:, 1] == trial]
        
        # Extraer solo canales EEG (columnas 4 a 67)
        eeg_data = trial_data.iloc[:, 4:68].values.T  # (64 canales, n_samples)
        
        if eeg_data.shape[1] < 256:  # Mínimo para calcular coherencia
            continue
        
        # Calcular matriz de coherencia
        try:
            coh_matrix = compute_coherence_matrix(eeg_data, FS, FREQ_BAND)
            matrices.append(coh_matrix)
        except Exception as e:
            continue
    
    return matrices

In [ ]:
# Crear carpetas
for split in ['Training', 'Test']:
    for n_channels in ['8', '16', '32', '64']:
        for label in ['HC', 'SZ']:
            path = os.path.join(OUTPUT_DIR, split, n_channels, label)
            os.makedirs(path, exist_ok=True)
            
print("Estructura de carpetas creada:")
print(f"  {OUTPUT_DIR}/")
print(f"    Training/")
print(f"    │    8/, 16/, 32/, 64/")
print(f"    │      └── HC/, SZ/")
print(f"    Test/")
print(f"         └── ...")

Estructura de carpetas creada:
  /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_Schizophrenia/
    ├── Training/
    │   ├── 8/, 16/, 32/, 64/
    │   │   └── HC/, SZ/
    └── Test/
        └── ...


In [8]:
def generate_and_save_matrices(subjects_dict, split_name, channel_configs=[64, 32, 16, 8]):
    """
    Genera y guarda matrices de conectividad para todos los sujetos.
    """
    counts = {ch: {'HC': 0, 'SZ': 0} for ch in channel_configs}
    
    for label, subjects in subjects_dict.items():
        print(f"\nProcesando {label} ({split_name})...")
        
        for subject_id in tqdm(subjects, desc=f"{label}"):
            # Obtener matrices 64x64
            matrices_64 = process_subject(int(subject_id), condition=CONDITION)
            
            if len(matrices_64) == 0:
                continue
            
            # Para cada configuración de canales
            for n_ch in channel_configs:
                # Seleccionar canales
                ch_indices = select_channels(n_ch, 64)
                
                for idx, mat_64 in enumerate(matrices_64):
                    # Extraer submatriz
                    mat_sub = mat_64[np.ix_(ch_indices, ch_indices)]
                    
                    # Guardar
                    filename = f"sub{subject_id:03d}_trial{idx:03d}.npy"
                    filepath = os.path.join(OUTPUT_DIR, split_name, str(n_ch), label, filename)
                    np.save(filepath, mat_sub)
                    counts[n_ch][label] += 1
    
    return counts

In [9]:
# Generar Training
print("="*60)
print("GENERANDO MATRICES DE TRAINING")
print("="*60)
train_counts = generate_and_save_matrices(train_subjects, 'Training')

GENERANDO MATRICES DE TRAINING

Procesando HC (Training)...


HC:   0%|          | 0/26 [00:00<?, ?it/s]

  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/64.csv/64.csv


HC:   8%|▊         | 2/26 [05:33<1:06:36, 166.51s/it]

  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/59.csv/59.csv


...

SZ:  82%|████████▏ | 32/39 [51:42<16:19, 139.96s/it]

  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/50.csv/50.csv


SZ:  87%|████████▋ | 34/39 [1:00:45<14:46, 177.24s/it]

  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/38.csv/38.csv
  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/49.csv/49.csv
  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/28.csv/28.csv
  Archivo no encontrado: /home/manu/TFG2/schizophrenia_dataset/42.csv/42.csv


SZ: 100%|██████████| 39/39 [1:09:05<00:00, 106.31s/it]


In [10]:
# GENERAR SOLO TEST
test_subjects_corrected = {
    'HC': [7, 8, 11, 15, 20],
    'SZ': [69, 74, 76]
}

print("="*60)
print("GENERANDO MATRICES DE TEST")
print("="*60)
test_counts = generate_and_save_matrices(test_subjects_corrected, 'Test')

print("\n--- TEST ---")
for ch in [64, 32, 16, 8]:
    total = test_counts[ch]['HC'] + test_counts[ch]['SZ']
    print(f"{ch} canales: HC={test_counts[ch]['HC']}, SZ={test_counts[ch]['SZ']}, Total={total}")

GENERANDO MATRICES DE TEST

Procesando HC (Test)...


HC: 100%|██████████| 5/5 [41:29<00:00, 497.89s/it]



Procesando SZ (Test)...


SZ: 100%|██████████| 3/3 [24:21<00:00, 487.00s/it]


--- TEST ---
64 canales: HC=474, SZ=291, Total=765
32 canales: HC=474, SZ=291, Total=765
16 canales: HC=474, SZ=291, Total=765
8 canales: HC=474, SZ=291, Total=765


In [ ]:
# Generar Test
print("="*60)
print("GENERANDO MATRICES DE TEST")
print("="*60)
test_counts = generate_and_save_matrices(test_subjects, 'Test')

In [11]:
print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)

print("\n--- TRAINING ---")
for ch in [64, 32, 16, 8]:
    total = train_counts[ch]['HC'] + train_counts[ch]['SZ']
    print(f"{ch} canales: HC={train_counts[ch]['HC']}, SZ={train_counts[ch]['SZ']}, Total={total}")

print("\n--- TEST ---")
for ch in [64, 32, 16, 8]:
    total = test_counts[ch]['HC'] + test_counts[ch]['SZ']
    print(f"{ch} canales: HC={test_counts[ch]['HC']}, SZ={test_counts[ch]['SZ']}, Total={total}")

print("\n" + "="*60)
print(f"Matrices guardadas en: {OUTPUT_DIR}")
print("="*60)


RESUMEN FINAL

--- TRAINING ---
64 canales: HC=1951, SZ=1160, Total=3111
32 canales: HC=1951, SZ=1160, Total=3111
16 canales: HC=1951, SZ=1160, Total=3111
8 canales: HC=1951, SZ=1160, Total=3111

--- TEST ---
64 canales: HC=474, SZ=291, Total=765
32 canales: HC=474, SZ=291, Total=765
16 canales: HC=474, SZ=291, Total=765
8 canales: HC=474, SZ=291, Total=765

Matrices guardadas en: /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_Schizophrenia
